# Bronze: DuoTrace application export

This notebook ingests completed DuoTrace searches from the protected Cloudflare Worker export endpoint. It does not call the Riot API directly.

In [ ]:
KEY_VAULT_URL = "https://your-key-vault-name.vault.azure.net/"
EXPORT_TOKEN_SECRET_NAME = "duotrace-analytics-export-token"
EXPORT_URL = "https://your-worker.your-subdomain.workers.dev/api/analytics/export"
PAGE_SIZE = 100
REQUEST_TIMEOUT_SECONDS = 30

In [ ]:
import json
import uuid
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, TimestampType

try:
    from notebookutils import notebookutils
except ImportError:
    from notebookutils import mssparkutils as notebookutils

if "your-key-vault-name" in KEY_VAULT_URL or "your-worker" in EXPORT_URL:
    raise ValueError("Set KEY_VAULT_URL and EXPORT_URL before running the notebook.")

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)
export_token = notebookutils.credentials.getSecret(KEY_VAULT_URL, EXPORT_TOKEN_SECRET_NAME)

In [ ]:
raw_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_sequence", IntegerType(), False),
    StructField("exported_at", TimestampType(), False),
    StructField("record_json", StringType(), False)
])
checkpoint_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("next_cursor", IntegerType(), False),
    StructField("updated_at", TimestampType(), False)
])
run_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("records_ingested", IntegerType(), False),
    StructField("next_cursor", IntegerType(), False),
    StructField("error_summary", StringType(), True)
])

cursor = 0
if spark.catalog.tableExists("ops_duotrace_export_checkpoint"):
    latest_checkpoint = spark.table("ops_duotrace_export_checkpoint").orderBy(F.desc("updated_at")).limit(1).collect()
    if latest_checkpoint:
        cursor = latest_checkpoint[0]["next_cursor"]

In [ ]:
headers = {"Authorization": f"Bearer {export_token}"}
records = []
has_more = True

while has_more:
    response = requests.get(
        EXPORT_URL,
        headers=headers,
        params={"after": cursor, "limit": PAGE_SIZE},
        timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()
    page = response.json()

    records.extend(page["records"])
    cursor = page["nextCursor"]
    has_more = page["hasMore"]

raw_rows = [
    (RUN_ID, record["sequence"], datetime.now(timezone.utc), json.dumps(record, separators=(",", ":")))
    for record in records
]

if raw_rows:
    spark.createDataFrame(raw_rows, raw_schema).write.format("delta").mode("append").saveAsTable("bronze_duotrace_search_raw")

spark.createDataFrame([(RUN_ID, cursor, datetime.now(timezone.utc))], checkpoint_schema).write.format("delta").mode("append").saveAsTable("ops_duotrace_export_checkpoint")
spark.createDataFrame([(RUN_ID, RUN_STARTED_AT, datetime.now(timezone.utc), "Succeeded", len(raw_rows), cursor, None)], run_log_schema).write.format("delta").mode("append").saveAsTable("ops_duotrace_export_run_log")

print({"run_id": RUN_ID, "records_ingested": len(raw_rows), "next_cursor": cursor})